In [13]:
import requests
import json

def fetch_char_list():
    url = "https://en.wiktionary.org/w/api.php"
    # The 'headers' part is what fixes your error
    headers = {"User-Agent": "PolywordProject (contact: Parama.Wattanakrai@example.com)"}
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": "Category:Chinese_hanzi",
        "cmlimit": "500",
        "format": "json"
    }
    
    char_list = []
    print("Fetching character names...")

    while True:
        response = requests.get(url, params=params, headers=headers)
        data = response.json()
        
        for member in data['query']['categorymembers']:
            char_list.append(member['title'])
        
        print(f"Collected {len(char_list)} names...", end="\r")
        
        if 'continue' in data:
            params['cmcontinue'] = data['continue']['cmcontinue']
        else:
            break

    with open("char_list.json", "w", encoding="utf-8") as f:
        json.dump(char_list, f, ensure_ascii=False)
    print(f"\nDone! Saved {len(char_list)} names to char_list.json")

fetch_char_list()

Fetching character names...
Collected 34363 names...
Done! Saved 34363 names to char_list.json


In [15]:
import requests
import re
import json
import time

def extract_data():
    with open("char_list.json", "r", encoding="utf-8") as f:
        char_list = json.load(f)

    url = "https://en.wiktionary.org/w/api.php"
    headers = {"User-Agent": "PolywordProject (contact: Parama.Wattanakrai@example.com)"}
    output_file = "han_compound_wiktionary.jsonl"

    print(f"Starting extraction for {len(char_list)} characters...")

    with open(output_file, "a", encoding="utf-8") as f:
        for index, char in enumerate(char_list, 1):
            params = {
                "action": "query",
                "prop": "revisions",
                "titles": char,
                "rvprop": "content",
                "format": "json"
            }
            
            try:
                response = requests.get(url, params=params, headers=headers).json()
                pages = response['query']['pages']
                
                compound_values = [] 
                char_value = ""
                zh_forms_value = ""
                
                for page_id in pages:
                    if "revisions" in pages[page_id]:
                        wikitext = pages[page_id]['revisions'][0]['*']
                        
                        char_match = re.search(r"\{\{Han char\|.*?\}\}", wikitext, re.DOTALL)
                        if char_match:
                            char_value = char_match.group(0).replace("\n", " ")

                        chinese_start = wikitext.find("==Chinese==")
                        if chinese_start != -1:
                            rest_of_text = wikitext[chinese_start + 11:]
                            next_lang = re.search(r"\n==[^=]", rest_of_text)
                            
                            if next_lang:
                                chinese_text = rest_of_text[:next_lang.start()]
                            else:
                                chinese_text = rest_of_text

                            zh_match = re.search(r"\{\{zh-forms\|.*?\}\}", chinese_text, re.DOTALL)
                            if zh_match:
                                zh_forms_value = zh_match.group(0).replace("\n", " ")

                            comp_matches = re.findall(r"\{\{Han compound\|.*?\}\}", chinese_text, re.DOTALL)
                            if len(comp_matches) > 1:
                                print(f"\n[Note] Found {len(comp_matches)} compounds for {char}")
                            
                            compound_values = [c.replace("\n", " ") for c in comp_matches]

                record = {
                    "character": char,
                    "zh_forms": zh_forms_value,
                    "han_compounds": compound_values,
                    "han_char": char_value
                }
                
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                f.flush() 

                if index % 10 == 0:
                    print(f"Processed: {index}/{len(char_list)}", end="\r")
                
                time.sleep(0.03)

            except Exception as e:
                print(f"\nError on {char}: {e}")
                f.write(json.dumps({"character": char, "error": str(e)}, ensure_ascii=False) + "\n")
                continue

    print(f"\nFinished! Data saved to {output_file}")

extract_data()

Starting extraction for 34363 characters...
Processed: 520/34363
[Note] Found 2 compounds for 亶
Processed: 600/34363
[Note] Found 2 compounds for 付
Processed: 790/34363
[Note] Found 2 compounds for 体

[Note] Found 2 compounds for 何
Processed: 800/34363
[Note] Found 2 compounds for 你
Processed: 1020/34363
[Note] Found 2 compounds for 便
Processed: 1040/34363
[Note] Found 2 compounds for 俛
Processed: 1150/34363
[Note] Found 2 compounds for 倌
Processed: 1180/34363
[Note] Found 2 compounds for 倦
Processed: 1550/34363
[Note] Found 2 compounds for 傷

[Note] Found 2 compounds for 傾
Processed: 1560/34363
[Note] Found 2 compounds for 僉
Processed: 1950/34363
[Note] Found 2 compounds for 兌
Processed: 2080/34363
[Note] Found 3 compounds for 再
Processed: 2130/34363
[Note] Found 2 compounds for 冠

[Note] Found 2 compounds for 冥
Processed: 2140/34363
[Note] Found 3 compounds for 冪
Processed: 2270/34363
[Note] Found 2 compounds for 凱
Processed: 2330/34363
[Note] Found 2 compounds for 列
Processed: 2600/